# go-evo: the network's output next to the board

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrimozRavbar/go-evo/blob/main/notebooks/games_policy_colab.ipynb)

Same as `games_click_colab`, but every board has a second board on its left showing the **last layer of the network**: the move probabilities (or the raw logits) of the player to move.

**Reading the left board**
* It shows the position the player saw *before* its move, coloured by its output; a **blue ring** marks the move it then played. The right board shows the position after the move (red dot).
* `prob`: softmax of the logits over the allowed moves at temperature 1, i.e. the network's own distribution (games may be played at another `TEMPERATURE`). Colour runs from 0 to the top probability. The 6 best empty points are labelled in percent (`0.7` = 0.7%).
* `log prob`: log10 of the same probabilities, fixed scale 10⁻⁴ … 1, which shows the small differences that `prob` hides.
* `logit`: the raw output for every point, legal or not (also under the stones), scale min … max; the 6 best empty points are labelled with their logit.
* A grey × marks an empty point that is not allowed (ko, suicide, or your own eye under the phase 1–5 rules).
* Below: the top move, the pass probability and logit, and the entropy of the distribution (exp(entropy) ≈ the number of moves it is spread over).
* The random mover and you have no network, so their moves show only the position.

19×19, komi 7.5, area scoring, move cap 1083. Two rule sets, chosen automatically:
* **Phase 6 (Chinese rules)**, used whenever a **PPO 6** or **PPO 6b** player is in the game: pass any time, two passes in a row end the game, filling your own eye is allowed.
* **Phases 1–5 (forced passing)**, used for all other games, as in their training: pass only when no other move is possible, never fill your own eye.

**Players**
* **PPO 6b 0–3**: phase 6 trained on to 36 h (175.8k games) at a lower learning rate, with bigger batches; beats PPO 6 25 of 32. **The strongest phase 6 player.**
* **PPO 6 0–3**: phase 6, the small network trained from scratch for 16 h under Chinese rules (96.6k games). PPO 6 and 6b are the only players trained to pass; the older players never learned when to pass.
* **PPO final 0–3**: the last 4 policies of the phase 5 PPO run (8 h of self-play).
* **PPO big 0–3**: phase 5b, the bigger PPO network (115k weights, 8 h); weaker than PPO final.
* **PPO 5c 0–3**: the same big network trained on to 20 h at a lower learning rate; beats PPO final 22 of 32.
* **PPO 5d 0–3**: the big network after 32 h; beats PPO 5c 21 of 32 and PPO final 19 of 32.
* **PPO 5e 0–3**: the big network after 48 h with bigger batches, **the strongest player so far**, though only just: beats PPO 5d 18 of 32 and PPO final 19 of 32.
* **Evo phase 4 #0–3**: the final parents of phase 4 evolution (same network).
* **Evo 2d conv** and **Evo 3 low-rank**: champions from earlier phases.
* **random mover**: picks uniformly among the allowed moves.
* **you**: type moves such as `D4`, or use the clickable board in section 1.

`TEMPERATURE = 0` plays each network's top move (deterministic, as in all evaluations). Use 0.2–0.5 to see different games between the same two players.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"
if IN_COLAB:
    if not os.path.exists("go-evo"):
        !git clone -q --depth 1 -b {BRANCH} https://github.com/PrimozRavbar/go-evo.git
    %cd go-evo
    !pip install -q numba
else:
    os.chdir(os.path.abspath(".."))  # repo root
    sys.path.insert(0, os.getcwd())

import numpy as np
from goevo.match import GameConfig, play_game
from goevo.player import Player, RandomPlayer, load_population
from goevo.viz import draw_board, plot_score
from goevo.policyview import (PolicyClickGame, play_game_with_policy, animate_policy,
                              policy_slider, watch_live_policy)
print("ready")

## Load the players

In [ ]:
class HumanPlayer:
    """Type moves like D4 or Q16 ('pass': any time under phase-6 rules, else only when no stone move is allowed)."""
    def __init__(self, name="you"):
        self.name = name
    def select_move(self, game, allowed, legal, max_moves, temperature=0.0, rng=None):
        n = game.board.shape[0]
        cols = "ABCDEFGHJKLMNOPQRST"[:n]
        while True:
            s = input(f"move {game.move_number + 1} ({'B' if game.to_play == 1 else 'W'}): ").strip().upper()
            if s in ("PASS", "P"):
                a = n * n
            else:
                try:
                    a = (n - int(s[1:])) * n + cols.index(s[0])
                except (ValueError, IndexError):
                    print("format: D4, Q16 or pass"); continue
            if 0 <= a <= n * n and allowed[a]:
                return a
            print("not allowed here (illegal; under phase 1-5 rules also own eye, or pass while stone moves exist)")

PLAYERS = {}
def add(label, path, k=4):
    if not os.path.exists(path):
        print("missing", path); return
    genomes, cfg, *_ = load_population(path)
    for i, g in enumerate(genomes[:k]):
        PLAYERS[f"{label} {i}" if label.startswith("PPO") else f"{label} #{i}"] = Player(g, cfg, name=f"{label} {i}")
add("PPO 7", "runs/p7_s0/final.pt")
add("PPO 6b", "runs/p6b_s0/final.pt")
add("PPO 6", "runs/p6_s0/final.pt")
add("PPO final", "runs/p5_s0/final.pt")
add("PPO big", "runs/p5b_s0/final.pt")
add("PPO 5c", "runs/p5c_s0/final.pt")
add("PPO 5d", "runs/p5d_s0/final.pt")
add("PPO 5e", "runs/p5e_s0/final.pt")
add("Evo phase 4", "runs/p4_s0/gen4000.pt")
add("Evo 2d conv", "runs/d_sel_s0/gen1000.pt")
add("Evo 3 low-rank", "runs/p3_r32_s0/gen10000.pt")
PLAYERS["random mover"] = RandomPlayer("random mover")
PLAYERS["you"] = HumanPlayer()
from dataclasses import replace
from goevo.match import CHINESE
def rules_for(*names, temperature=0.0):
    """Chinese rules if a phase-6 or phase-7 player is in the game, else the phase 1-5 forced-pass rules."""
    base = CHINESE if any(n.startswith(("PPO 6", "PPO 7")) for n in names) else GameConfig(board_size=19)
    return replace(base, temperature=temperature)
print(list(PLAYERS))

## 1. Play by clicking, with the network's output on the left
Same board and rules as in `games_click_colab` (see there for Pass, End and score, Show area).
* **Show** picks `prob`, `log prob` or `logit`; you can switch during the game.
* **for**: *net's last move* shows the output the network had when it chose its last move (on the position it saw, blue ring on its move). *your move now* shows what the same network would play in **your** current position, from your side of the board.


In [ ]:
OPPONENT = "PPO 5d 0"  # @param ["PPO 7 0", "PPO 7 1", "PPO 7 2", "PPO 7 3", "PPO 6b 0", "PPO 6b 1", "PPO 6b 2", "PPO 6b 3", "PPO 6 0", "PPO 6 1", "PPO 6 2", "PPO 6 3", "PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO 5e 0", "PPO 5e 1", "PPO 5e 2", "PPO 5e 3", "PPO 5d 0", "PPO 5d 1", "PPO 5d 2", "PPO 5d 3", "PPO 5c 0", "PPO 5c 1", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover"]
YOU_PLAY = "black"  # @param ["black", "white"]
TEMPERATURE = 0.0  # @param {type:"number"}
SHOW = "prob"  # @param ["prob", "log prob", "logit"]
cg = PolicyClickGame(PLAYERS[OPPONENT], human_color=1 if YOU_PLAY == "black" else -1, cfg=rules_for(OPPONENT),
                     temperature=TEMPERATURE, mode=SHOW).show()

## 2. Play one game and replay it
Pick the players and run the cell. Each frame is one move: left, the mover's output before the move (blue ring on the move played); right, the board after it with the running count. `EVERY` shows every k-th move. The animation has twice as many boards as in `games_click_colab`, so it takes longer to build; raise `EVERY` for long games.


In [ ]:
BLACK = "PPO final 0"  # @param ["PPO 7 0", "PPO 7 1", "PPO 7 2", "PPO 7 3", "PPO 6b 0", "PPO 6b 1", "PPO 6b 2", "PPO 6b 3", "PPO 6 0", "PPO 6 1", "PPO 6 2", "PPO 6 3", "PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO 5e 0", "PPO 5e 1", "PPO 5e 2", "PPO 5e 3", "PPO 5d 0", "PPO 5d 1", "PPO 5d 2", "PPO 5d 3", "PPO 5c 0", "PPO 5c 1", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
WHITE = "Evo phase 4 #0"  # @param ["PPO 7 0", "PPO 7 1", "PPO 7 2", "PPO 7 3", "PPO 6b 0", "PPO 6b 1", "PPO 6b 2", "PPO 6b 3", "PPO 6 0", "PPO 6 1", "PPO 6 2", "PPO 6 3", "PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO 5e 0", "PPO 5e 1", "PPO 5e 2", "PPO 5e 3", "PPO 5d 0", "PPO 5d 1", "PPO 5d 2", "PPO 5d 3", "PPO 5c 0", "PPO 5c 1", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
TEMPERATURE = 0.0  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}
EVERY = 2  # @param {type:"integer"}
SHOW = "prob"  # @param ["prob", "log prob", "logit"]

gc = rules_for(BLACK, WHITE, temperature=TEMPERATURE)
rec = play_game_with_policy(PLAYERS[BLACK], PLAYERS[WHITE], gc, np.random.default_rng(SEED))
print(f"{BLACK} (B) vs {WHITE} (W): {rec.result_string}, {len(rec.moves)} moves")
animate_policy(rec, interval_ms=300, every=EVERY, mode=SHOW)

### Scrub through the same game move by move
The drop-down switches between `prob`, `log prob` and `logit`.

In [ ]:
policy_slider(rec, mode=SHOW)


### Final position and score
The final count, and how the running margin moved during the game (above 0: Black ahead).

In [ ]:
import matplotlib.pyplot as plt
draw_board(rec.boards[-1], last_move=rec.moves[-1], title=f"final: {rec.result_string}", score=True, komi=rec.komi)
plt.show()
plot_score(rec)
plt.show()
print(rec.to_sgf()[:300], "...")  # full SGF in rec.to_sgf(), can be opened in any Go viewer

## 3. Watch live
After every move: left, the mover's output before it; right, the board. Set `DELAY` in seconds. To play yourself, set one side to `you` and type moves in the input box (your moves show no output; the network's do).


In [ ]:
BLACK = "you"  # @param ["PPO 7 0", "PPO 7 1", "PPO 7 2", "PPO 7 3", "PPO 6b 0", "PPO 6b 1", "PPO 6b 2", "PPO 6b 3", "PPO 6 0", "PPO 6 1", "PPO 6 2", "PPO 6 3", "PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO 5e 0", "PPO 5e 1", "PPO 5e 2", "PPO 5e 3", "PPO 5d 0", "PPO 5d 1", "PPO 5d 2", "PPO 5d 3", "PPO 5c 0", "PPO 5c 1", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
WHITE = "PPO final 0"  # @param ["PPO 7 0", "PPO 7 1", "PPO 7 2", "PPO 7 3", "PPO 6b 0", "PPO 6b 1", "PPO 6b 2", "PPO 6b 3", "PPO 6 0", "PPO 6 1", "PPO 6 2", "PPO 6 3", "PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO 5e 0", "PPO 5e 1", "PPO 5e 2", "PPO 5e 3", "PPO 5d 0", "PPO 5d 1", "PPO 5d 2", "PPO 5d 3", "PPO 5c 0", "PPO 5c 1", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
TEMPERATURE = 0.0  # @param {type:"number"}
DELAY = 0.05  # @param {type:"number"}
SHOW = "prob"  # @param ["prob", "log prob", "logit"]
live = watch_live_policy(PLAYERS[BLACK], PLAYERS[WHITE], rules_for(BLACK, WHITE, temperature=TEMPERATURE),
                         delay=DELAY, rng=np.random.default_rng(1), mode=SHOW)